# Nano Urdu evaluation with Whisper large-v3

This notebook measures how **intelligible** the Nano Urdu voice is. It speaks
held-out sentences, has OpenAI Whisper large-v3 transcribe the audio back to Urdu
text, and scores the transcript against the sentence that was spoken:

* **WER** — word error rate, and **CER** — character error rate (lower is better).

Whisper makes Urdu mistakes of its own, so a WER on its own means little. The
notebook therefore scores three sources through the same pipeline:

| System | What it is | What it tells you |
|---|---|---|
| **human** | real recordings of the same test sentences | the floor: how well Whisper can do at all |
| **teacher** | MMS-TTS Urdu, the voice Nano Urdu learned from | the ceiling for the small model |
| **nano** | your 443,402-parameter int8 voice, run through the shipped C runtime | the model under test |

Read the **nano** numbers against the **teacher** and **human** rows, not against zero.

Two test sets are used:

* **corpus** — sentences from the `test` split of `kaab4321/UrduTTS` that the model
  never trained on (the training notebook only used its `train` and `eval` splits).
  These have human recordings, so all three systems are scored.
* **everyday** — 47 hand-written sentences far from the corpus (greetings, facts,
  numbers, English loanwords, short phrases, long sentences). No recordings, so
  only nano and teacher are scored.

## How to run

1. Nothing to upload: the v7 voice (443,402 parameters) is built into this
   notebook. To test a different voice, upload its `nano-urdu-pilot-*.zip` as a
   **Kaggle dataset** and add it with **Add Input** (it is found under `/kaggle/input`
   and used instead), or set `VOICE_PATH` below.
2. Enable a **GPU** (Whisper large-v3 needs one) and **Internet**, then run the cells in order.
3. Download `nano-urdu-eval-*.zip` from the output: the summary, every transcript,
   a chart, and the audio of the worst sentences.

Set `SMOKE_TEST = True` for a quick plumbing check with a tiny Whisper
(its scores are meaningless).

[Whisper large-v3 (MIT)](https://huggingface.co/openai/whisper-large-v3) ·
[MMS-TTS Urdu (CC BY-NC 4.0)](https://huggingface.co/facebook/mms-tts-urd-script_arabic) ·
[UrduTTS corpus (CC BY-NC 4.0)](https://huggingface.co/datasets/kaab4321/UrduTTS)

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

NOTEBOOK_VERSION = "eval-2 (Whisper large-v3 WER/CER, built-in v7 voice)"
print("Nano Urdu evaluation notebook", NOTEBOOK_VERSION)

SMOKE_TEST = False
WHISPER_MODEL = "openai/whisper-large-v3"
CORPUS_SENTENCES = 200      # held-out test-split sentences that have human recordings
EVAL_TEACHER = True         # also score the MMS-TTS teacher (the ceiling for the small model)
EVAL_HUMAN = True           # also score the real recordings (the floor set by Whisper itself)
WHISPER_BATCH = 16
WHISPER_BEAMS = 1           # 1 = greedy (fast); 5 is a little more accurate and ~3x slower
TEACHER_NOISE_SCALE = 0.1   # same randomness the teacher had when it made the training data
VOICE_PATH = ""             # zip or folder with another voice; empty = an added input, else the built-in v7
RUN_NAME = "eval-v7"
SEED = 0

if SMOKE_TEST:
    RUN_NAME += "-smoke"
    WHISPER_MODEL = "openai/whisper-tiny"
    CORPUS_SENTENCES = 6
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use letters, numbers, underscores, or hyphens in RUN_NAME.")

ON_KAGGLE = Path("/kaggle/working").exists()
WORKING = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("NANO_EVAL_WORKDIR", "eval-work")).resolve()
PROJECT = Path("/kaggle/temp/nanourdu-eval") if ON_KAGGLE else WORKING / "project"
OUT = WORKING / "nano-urdu-eval" / RUN_NAME
for directory in (PROJECT, OUT):
    directory.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")

def run(*args, check=True, cwd=None):
    command = [str(arg) for arg in args]
    print("+ " + " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd or PROJECT, env=ENV, check=check)

In [ ]:
# Embedded Nano Urdu source (the same text front end the model was trained with)

SOURCES = {
    'phones.py': '"""Shared IPA segmentation for corpus alignment and synthesis."""\nimport re\nimport unicodedata\n\nBOUNDARY = "|"  # word-boundary / pause token; carries the silence between words\nLENGTH = "ː"\n\n\ndef phone_pattern(inventory):\n    symbols = sorted((phone for phone in inventory\n                      if phone and not phone.startswith("<") and phone != BOUNDARY),\n                     key=lambda phone: (-len(phone), phone))\n    if not symbols:\n        raise ValueError("The phone inventory is empty.")\n    return re.compile("|".join(map(re.escape, symbols)))\n\n\ndef tokenize_words(ipa, pattern):\n    """IPA split into words, each a list of phones."""\n    ipa = unicodedata.normalize("NFC", ipa).translate(str.maketrans("", "", "ˈˌ͜͡\u200c\u200d"))\n    words = []\n    for word in re.split(r"[\\s.,:;!?،؛؟۔…“”|]+", ipa):\n        phones, offset = [], 0\n        while offset < len(word):\n            match = pattern.match(word, offset)\n            if match is None:\n                symbol = word[offset]\n                if symbol == LENGTH and phones:\n                    # A length mark the inventory has no combined symbol for\n                    # (eSpeak writes Urdu geminates as e.g. "nː"): hold the\n                    # previous phone for twice as long by repeating it.\n                    phones.append(phones[-1])\n                    offset += 1\n                    continue\n                if unicodedata.combining(symbol):\n                    offset += 1  # an unsupported diacritic (e.g. stray nasal tilde)\n                    continue\n                raise ValueError(f"unknown IPA at {word[offset:]!r}")\n            phones.append(match[0])\n            offset = match.end()\n        if phones:\n            words.append(phones)\n    return words\n\n\ndef tokenize(ipa, pattern):\n    return [phone for word in tokenize_words(ipa, pattern) for phone in word]\n\n\ndef with_boundaries(words):\n    """Phones with a boundary token before, between, and after words."""\n    sequence = [BOUNDARY]\n    for word in words:\n        sequence.extend(word)\n        sequence.append(BOUNDARY)\n    return sequence\n',
    'urdu_frontend.py': '"""Urdu-script text and number normalization for Nano Urdu."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport unicodedata\n\nfrom phones import BOUNDARY, LENGTH, phone_pattern, tokenize_words, with_boundaries\n\n\nCARDINALS = """\nصفر ایک دو تین چار پانچ چھ سات آٹھ نو\nدس گیارہ بارہ تیرہ چودہ پندرہ سولہ سترہ اٹھارہ انیس\nبیس اکیس بائیس تئیس چوبیس پچیس چھبیس ستائیس اٹھائیس انتیس\nتیس اکتیس بتیس تینتیس چونتیس پینتیس چھتیس سینتیس اڑتیس انتالیس\nچالیس اکتالیس بیالیس تینتالیس چوالیس پینتالیس چھیالیس سینتالیس اڑتالیس انچاس\nپچاس اکیاون باون ترپن چوّن پچپن چھپن ستاون اٹھاون انسٹھ\nساٹھ اکسٹھ باسٹھ تریسٹھ چونسٹھ پینسٹھ چھیاسٹھ سڑسٹھ اڑسٹھ انہتر\nستر اکہتر بہتر تہتر چوہتر پچہتر چھہتر ستتر اٹھہتر اناسی\nاسی اکیاسی بیاسی تراسی چوراسی پچاسی چھیاسی ستاسی اٹھاسی نواسی\nنوے اکانوے بانوے ترانوے چورانوے پچانوے چھیانوے ستانوے اٹھانوے ننانوے\n""".split()\nSCALES = ((10**12, "کھرب"), (10**9, "ارب"), (10**7, "کروڑ"),\n          (10**5, "لاکھ"), (1000, "ہزار"), (100, "سو"))\nDIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩كي−٫٬٪",\n                          "01234567890123456789کی-.,%")\nNUMBER = re.compile(r"[+-]?[0-9]+(?:,[0-9]+)*(?:\\.[0-9]+)?(?:[ \\t]*%)?")\nGROUPING = re.compile(r"(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]{1,2}(?:,[0-9]{2})+,[0-9]{3})")\nPUNCTUATION = "۔،؛؟!?.,:;…()«»\\"\'“”–—-"\n\n\ndef cardinal(number):\n    words = []\n    for scale, name in SCALES:\n        units, number = divmod(number, scale)\n        if units:\n            words.extend((CARDINALS[units], name))\n    if number or not words:\n        words.append(CARDINALS[number])\n    return " ".join(words)\n\n\ndef spell_digits(number):\n    return " ".join(CARDINALS[int(digit)] for digit in number)\n\n\ndef speak_number(match, digit_by_digit):\n    token = match[0]\n    sign = {"-": "منفی", "+": "مثبت"}.get(token[0])\n    percent = token.endswith("%")\n    unsigned = token.lstrip("+-").removesuffix("%").strip()\n    integer, _, fraction = unsigned.partition(".")\n    if "," in integer and not GROUPING.fullmatch(integer):\n        raise ValueError(f"Invalid digit grouping in {token!r}. Use spaces between separate numbers.")\n    integer = integer.replace(",", "")\n    if digit_by_digit or len(integer) > 14 or (len(integer) > 1 and integer.startswith("0")):\n        words = [spell_digits(integer)]\n    else:\n        words = [cardinal(int(integer))]\n    if sign:\n        words.insert(0, sign)\n    if fraction:\n        words.extend(("اعشاریہ", spell_digits(fraction)))\n    if percent:\n        words.append("فیصد")\n    return " " + " ".join(words) + " "\n\n\ndef normalize(text, *, digit_by_digit=False):\n    text = unicodedata.normalize("NFKC", text).translate(DIGIT_MAP)\n    text = "".join(char for char in text if unicodedata.category(char) != "Cf" and char != "ـ")\n    text = re.sub(r"[0-9]+(?:،[0-9]+)+", normalize_grouping, text)\n    if re.search(r"[0-9]\\s*[:/–—-]\\s*[0-9]", text):\n        raise ValueError("Write times, fractions, dates, and ranges out in Urdu words.")\n    text = NUMBER.sub(lambda match: speak_number(match, digit_by_digit), text).strip()\n    for char in text:\n        if char.isspace() or char in PUNCTUATION:\n            continue\n        kind = unicodedata.category(char)[0]\n        if kind not in "LM" or not unicodedata.name(char, "").startswith("ARABIC"):\n            raise ValueError(f"Use Urdu script and numbers. Unsupported character {char!r}.")\n    if not any(unicodedata.category(char).startswith("L") for char in text):\n        raise ValueError("Enter some Urdu text or a number.")\n    return re.sub(r"\\s+", " ", text)\n\n\ndef normalize_grouping(match):\n    grouped = match[0].replace("،", ",")\n    return grouped if GROUPING.fullmatch(grouped) else match[0]\n\n\ndef phonemize(text):\n    executable = shutil.which("espeak-ng")\n    if executable is None:\n        raise RuntimeError("Install eSpeak NG for Urdu phonemization.")\n    result = subprocess.run([executable, "-q", "--ipa", "-v", "ur", "--stdin"],\n                            input=text + "\\n", text=True, capture_output=True, check=True)\n    return result.stdout.strip()\n\n\nVOWELS = "aeiouɛɔəɪʊʌɐ"\n# Nearest sounds for symbols eSpeak can emit (mostly in loanwords) that a voice\n# may never have seen in training; used only when the voice lacks the symbol.\nNEAREST = {"ʒ": "z", "g": "ɡ", "ŋ": "n", "ɲ": "n", "ɳ": "n", "ɭ": "l", "v": "ʋ",\n           "ɦ": "h", "χ": "x", "ʁ": "ɣ", "ɑ": "aː", "ɑː": "aː", "æ": "ɛ", "ɒ": "ɔ"}\n\n\ndef phone_aliases(vocabulary):\n    """Map phones the voice never saw to the nearest one it knows (e.g. loanword "o" -> "oː")."""\n    aliases = {source: target for source, target in NEAREST.items()\n               if source not in vocabulary and target in vocabulary}\n    for vowel in VOWELS:\n        for base in (vowel, vowel + "̃"):  # plain and nasal\n            short, long = unicodedata.normalize("NFC", base), unicodedata.normalize("NFC", base + LENGTH)\n            if short not in vocabulary and long in vocabulary:\n                aliases[short] = long\n            elif long not in vocabulary and short in vocabulary:\n                aliases[long] = short\n    return aliases\n\n\ndef phones_for(ipa, vocabulary):\n    aliases = phone_aliases(vocabulary)\n    words = tokenize_words(ipa, phone_pattern(set(vocabulary) | set(aliases)))\n    words = [[aliases.get(phone, phone) for phone in word] for word in words]\n    if not words:\n        raise ValueError("The phonemizer produced no usable phones.")\n    if BOUNDARY in vocabulary:  # voices trained with word-boundary / pause tokens\n        return with_boundaries(words)\n    return [phone for word in words for phone in word]\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("package", type=Path)\n    parser.add_argument("text")\n    parser.add_argument("output", type=Path)\n    parser.add_argument("--digits", action="store_true", help="Read every integer digit individually.")\n    args = parser.parse_args()\n    try:\n        spoken = normalize(args.text, digit_by_digit=args.digits)\n        vocabulary = json.loads((args.package / "phones.json").read_text(encoding="utf-8"))\n        ipa = phonemize(spoken)\n        phones = phones_for(ipa, vocabulary)\n    except (ValueError, RuntimeError, subprocess.CalledProcessError) as error:\n        parser.error(str(error))\n    print(f"Spoken Urdu: {spoken}\\nIPA: {ipa}", flush=True)\n    subprocess.run([sys.executable, str(Path(__file__).with_name("speak.py")),\n                    str(args.package), " ".join(phones), str(args.output)], check=True)\n',
    'prepare_data.py': '"""Stream a bounded UrduTTS sample for audition and alignment."""\nimport argparse\nimport io\nimport json\nimport time\nfrom math import gcd\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport pyarrow as pa\nimport requests\nfrom scipy.signal import resample_poly\nimport soundfile as sf\n\n\nREPO = "kaab4321/UrduTTS"\nREVISION = "f5c50b7aaabd4325bfe33617232203ad02e30f97"\nBASE = f"https://huggingface.co/datasets/{REPO}/resolve/{REVISION}"\nSAMPLE_RATE = 24_000\n\n\ndef download(client, url, path, attempts=8):\n    """Fetch url to path, resuming after dropped connections."""\n    for attempt in range(attempts):\n        done = path.stat().st_size if path.exists() else 0\n        headers = {"Range": f"bytes={done}-"} if done else {}\n        try:\n            with client.get(url, stream=True, timeout=(30, 120), headers=headers) as response:\n                if response.status_code == 416:  # already complete\n                    return\n                response.raise_for_status()\n                mode = "ab" if done and response.status_code == 206 else "wb"\n                with path.open(mode) as stream:\n                    for chunk in response.iter_content(1 << 20):\n                        stream.write(chunk)\n            return\n        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as error:\n            wait = min(60, 5 * 2 ** attempt)\n            print(f"Download interrupted ({error.__class__.__name__}); retrying in {wait}s", flush=True)\n            time.sleep(wait)\n    raise RuntimeError(f"Could not download {url} after {attempts} attempts")\n\n\ndef rows(split):\n    """Yield corpus rows. Each shard is downloaded to disk first, so a slow\n    consumer (e.g. teacher synthesis) never holds a network stream open."""\n    with requests.Session() as client, TemporaryDirectory(prefix="nanourdu-shard-") as temp:\n        response = client.get(f"{BASE}/{split}/state.json", timeout=60)\n        response.raise_for_status()\n        for shard in response.json()["_data_files"]:\n            name = shard["filename"]\n            local = Path(temp) / Path(name).name\n            download(client, f"{BASE}/{split}/{name}", local)\n            try:\n                with pa.memory_map(str(local)) as source, pa.ipc.open_stream(source) as reader:\n                    index = 0\n                    for batch in reader:\n                        audio = batch["audio"]\n                        script = batch["transcription"]\n                        ipa = batch["phonemes"]\n                        for row in range(batch.num_rows):\n                            yield name, index, script[row].as_py(), ipa[row].as_py(), audio[row].as_py()\n                            index += 1\n            finally:\n                local.unlink(missing_ok=True)\n\n\ndef prepare(split, limit, output):\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with TemporaryDirectory(dir=output.parent, prefix=".nanourdu-") as temp:\n        staging = Path(temp) / "sample"\n        (staging / "wavs").mkdir(parents=True)\n        count = 0\n        source = rows(split)\n        try:\n            with (staging / "utterances.jsonl").open("w", encoding="utf-8") as manifest:\n                for shard, index, script, ipa, audio in source:\n                    if not script or not ipa or not audio or not audio["bytes"]:\n                        continue\n                    waveform, rate = sf.read(io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True)\n                    if not 1.5 <= len(waveform) / rate <= 12:\n                        continue\n                    divisor = gcd(rate, SAMPLE_RATE)\n                    mono = resample_poly(waveform.mean(axis=1), SAMPLE_RATE // divisor, rate // divisor)\n                    clip_id = f"{split}-{count:05d}"\n                    relative = f"wavs/{clip_id}.wav"\n                    sf.write(staging / relative, mono.clip(-1, 1), SAMPLE_RATE, subtype="PCM_16")\n                    manifest.write(json.dumps({\n                        "id": clip_id, "audio": relative, "transcription": script, "phonemes": ipa,\n                        "source_file": audio["path"], "source_split": split,\n                        "source_shard": shard, "source_row": index, "source_revision": REVISION,\n                    }, ensure_ascii=False) + "\\n")\n                    count += 1\n                    if count == limit:\n                        break\n        finally:\n            source.close()\n        if count < limit:\n            raise RuntimeError(f"Expected {limit} usable clips and found {count}")\n        staging.rename(output)\n    print(f"Saved {count} clips and their Urdu script and IPA to {output}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", choices=("train", "eval", "test"), default="train")\n    parser.add_argument("--limit", type=int, default=16)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    if args.limit < 1 or args.output.exists():\n        parser.error("provide a positive clip limit and a new output directory")\n    prepare(args.split, args.limit, args.output)\n',
}
SOURCE_DIGEST = 'f4a7cbdcb68c7a60b53e9ecffa0e65e9dc11dea9ad88d29a0e775bc6036f3f86'
for name, content in SOURCES.items():
    (PROJECT / name).write_text(content, encoding='utf-8')

# The v7 voice (443,402 parameters, int8), used when no voice is added as an input.
EMBEDDED_VOICE_NAME = 'nano-urdu-pilot-v7'
EMBEDDED_VOICE_B64 = (
    '...'
)

## Set up

Installs the dependencies, eSpeak NG (the pronunciation front end the voice was
trained with), and the pinned sanoTTS C runtime, then finds the voice files.

In [ ]:
SANOTTS_REV = "55c6f39b2b535b5a591667a0852a6c2778320fab"

if ON_KAGGLE:
    run(sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.57.6",
        "huggingface_hub>=0.34,<1", "pyarrow", "requests", "scipy", "soundfile", "accelerate")
if not shutil.which("espeak-ng"):
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-y", "-qq", "espeak-ng")
if not shutil.which("cc"):
    raise RuntimeError("A C compiler (cc) is needed to build the int8 runtime.")

# The sanoTTS runtime sources (MIT), pinned to the revision the voice was exported against.
VENDOR = PROJECT / "vendor" / "sanoTTS"
if not (VENDOR / ".git").exists():
    VENDOR.parent.mkdir(parents=True, exist_ok=True)
    checkout = Path(tempfile.mkdtemp(dir=VENDOR.parent, prefix="sanoTTS."))
    run("git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "--no-checkout",
        "https://github.com/Ampixa/sanoTTS.git", checkout)
    run("git", "-C", checkout, "sparse-checkout", "set", "mcu/src", "mcu/include", "mcu/ports/host")
    run("git", "-C", checkout, "fetch", "--depth", "1", "origin", SANOTTS_REV)
    run("git", "-C", checkout, "checkout", "--detach", "FETCH_HEAD")
    checkout.rename(VENDOR)
head = subprocess.run(["git", "-C", str(VENDOR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if head != SANOTTS_REV:
    raise RuntimeError(f"Expected sanoTTS at {SANOTTS_REV}, found {head}")
MCU = VENDOR / "mcu"

In [ ]:
def voice_dirs(root):
    """Folders under root that look like an exported int8 voice."""
    for blob in sorted(root.rglob("front_q8.bin")):
        folder = blob.parent
        if all((folder / name).is_file() for name in ("model_q8.bin", "phones.json", "export-report.json", "nano_q8_meta.h")):
            yield folder

def show_tree(root, depth=4, limit=60):
    """Print what is under root, so a missing voice is easy to diagnose."""
    if not root.exists():
        print(f"  {root} does not exist")
        return
    shown = 0
    for folder, folders, names in os.walk(root):
        level = Path(folder).relative_to(root).parts
        if len(level) >= depth:
            folders[:] = []
        for name in sorted(folders) + sorted(names):
            if shown == limit:
                print("  ...")
                return
            path = Path(folder) / name
            size = f"  ({path.stat().st_size:,} bytes)" if path.is_file() else "/"
            print("  " + str(path) + size)
            shown += 1
    if not shown:
        print(f"  {root} is empty")

def find_voice():
    roots = [Path(VOICE_PATH)] if VOICE_PATH else [Path("/kaggle/input"), WORKING]
    found = []
    for root in roots:
        if root.is_file() and root.suffix == ".zip":
            zips = [root]
        elif root.is_dir():
            found += list(voice_dirs(root))
            zips = sorted(root.rglob("*.zip"))
        else:
            zips = []
        for archive in zips:
            try:
                with zipfile.ZipFile(archive) as bundle:
                    if any(name.endswith("int8/front_q8.bin") for name in bundle.namelist()):
                        target = WORKING / "voice" / archive.stem
                        if not list(voice_dirs(target)) if target.exists() else True:
                            bundle.extractall(target)
                        found += list(voice_dirs(target))
            except zipfile.BadZipFile:
                print("Skipping a file that is not a valid zip:", archive)
    if not found and not VOICE_PATH:
        import base64, io
        target = WORKING / "voice" / EMBEDDED_VOICE_NAME / "int8"
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_VOICE_B64))) as bundle:
            bundle.extractall(target)
        print(f"No voice added as an input; using the {EMBEDDED_VOICE_NAME} voice built into this notebook.")
        found = [target]
    if not found:
        print("Looked for int8/front_q8.bin (with model_q8.bin, phones.json, export-report.json and "
              "nano_q8_meta.h) in a folder or a .zip under:", ", ".join(map(str, roots)))
        for root in roots:
            print(f"Contents of {root}:")
            show_tree(root)
        raise FileNotFoundError(
            "No Nano Urdu voice found. In the notebook sidebar choose Add Input, upload the "
            "nano-urdu-pilot-v7.zip from your training run as a dataset, add it, and run this cell again. "
            "Or set VOICE_PATH at the top to the zip or its int8 folder.")
    if len(found) > 1:
        print("Several voices found; using the last one:\n  " + "\n  ".join(map(str, found)))
    return found[-1]

VOICE = find_voice()
EXPORT = json.loads((VOICE / "export-report.json").read_text())
if EXPORT.get("weights") != "int8":
    raise ValueError("Expected an int8 export")
VOCAB = json.loads((VOICE / "phones.json").read_text(encoding="utf-8"))
MAX_TOKENS = EXPORT["shapes"]["NANO_DUR_MAX_TOKENS"]
VOICE_SHA = hashlib.sha256(b"".join((VOICE / n).read_bytes() for n in ("front_q8.bin", "model_q8.bin"))).hexdigest()[:16]
print(f"Voice: {VOICE}\n  {len(VOCAB)} phones, up to {MAX_TOKENS} tokens per call, weights sha256 {VOICE_SHA}")

In [ ]:
import ctypes as ct
import numpy as np

class NanoVoice:
    """The shipped sanoTTS int8 runtime (C, compiled once) driven through ctypes.

    Same call sequence and noise seeding as the training project's speak.py.
    """
    PCM = ct.CFUNCTYPE(ct.c_int, ct.POINTER(ct.c_float), ct.c_int, ct.c_void_p)

    class Config(ct.Structure):
        _fields_ = [("front_blob", ct.c_void_p), ("dec_blob", ct.c_void_p), ("arena", ct.c_void_p),
                    ("arena_size", ct.c_size_t), ("dur_override", ct.c_void_p), ("noise_seed", ct.c_uint64)]

    def __init__(self, package, mcu, workdir):
        report = json.loads((package / "export-report.json").read_text())
        self.shapes = report["shapes"]
        self.blobs = []
        for name in ("front_q8.bin", "model_q8.bin"):
            payload = (package / name).read_bytes()
            if len(payload) != report["bytes"][name]:
                raise ValueError(f"Incomplete weight blob {name}")
            self.blobs.append(ct.create_string_buffer(payload))
        library = workdir / "sano-nano.so"
        subprocess.run(["cc", "-shared", "-fPIC", "-O2", "-std=c99", "-DFSD_FAST_MATH",
                        f"-I{mcu / 'include'}", f"-I{mcu / 'src'}", f"-I{package.resolve()}",
                        str(mcu / "src/snt_nano.c"), str(mcu / "src/snt_kernels_ref.c"),
                        str(mcu / "ports/host/snt_port_host.c"), "-lm", "-o", str(library)], check=True)
        self.lib = ct.CDLL(str(library))
        self.lib.snt_nano_synthesize.argtypes = [ct.POINTER(self.Config), ct.POINTER(ct.c_int32), ct.c_int,
                                                  self.PCM, ct.c_void_p, ct.c_void_p]
        self.lib.snt_nano_synthesize.restype = ct.c_int

    def say(self, ids, seed):
        if not 2 <= len(ids) <= self.shapes["NANO_DUR_MAX_TOKENS"]:
            raise ValueError("phone count outside the voice's range")
        arena_size = 1024 * 1024 + 1024 * len(ids) * self.shapes["NANO_DUR_MAX_DURATION"]
        arena = ct.create_string_buffer(arena_size)
        config = self.Config(ct.addressof(self.blobs[0]), ct.addressof(self.blobs[1]), ct.addressof(arena),
                             arena_size, None, seed)
        chunks = []

        @self.PCM
        def collect(pcm, length, _user):
            chunks.append(np.ctypeslib.as_array(pcm, (length,)).copy())
            return 0

        tokens = (ct.c_int32 * len(ids))(*ids)
        status = self.lib.snt_nano_synthesize(ct.byref(config), tokens, len(ids), collect, None, None)
        if status != 0 or not chunks:
            raise RuntimeError(f"sanoTTS synthesis returned {status}")
        return np.concatenate(chunks)

sys.path.insert(0, str(PROJECT))
from urdu_frontend import normalize, phonemize, phones_for
import prepare_data

NANO = NanoVoice(VOICE, MCU, OUT)
SAMPLE_RATE = 24_000

## Test sentences

**corpus**: the first `CORPUS_SENTENCES` usable sentences of the corpus `test`
split (the shard is downloaded once, resuming if the connection drops). A sentence
is usable when the text normalizer accepts it, the voice can say it in one call,
and its recording lasts 1–12 s.

**everyday**: 47 sentences written for this notebook, with a category for each.
Numbers are spelled out in Urdu words by the same normalizer the voice uses, and
the transcripts are compared against that spoken form.

In [ ]:
import io
from math import gcd
from scipy.signal import resample_poly
import soundfile as sf

EVERYDAY = [
    ("daily", "السلام علیکم، آپ کیسے ہیں؟"),
    ("daily", "میں ٹھیک ہوں، شکریہ۔"),
    ("daily", "آپ کا نام کیا ہے؟"),
    ("daily", "میرا نام احمد ہے اور میں لاہور میں رہتا ہوں۔"),
    ("daily", "آج موسم بہت اچھا ہے، چلو باہر چلتے ہیں۔"),
    ("daily", "کل رات بہت تیز بارش ہوئی تھی۔"),
    ("daily", "مجھے چائے بہت پسند ہے لیکن کافی نہیں۔"),
    ("daily", "کیا آپ مجھے ریلوے اسٹیشن کا راستہ بتا سکتے ہیں؟"),
    ("daily", "براہ کرم دروازہ بند کر دیں۔"),
    ("daily", "بچے اسکول کے میدان میں کرکٹ کھیل رہے ہیں۔"),
    ("daily", "ہم سب کل شام کو دعوت پر جائیں گے۔"),
    ("daily", "امی نے آج بہت مزیدار کھانا بنایا ہے۔"),
    ("daily", "مجھے ایک گلاس پانی دے دیجیے۔"),
    ("daily", "یہ کتاب بہت دلچسپ ہے، میں نے ایک ہی دن میں ختم کر لی۔"),
    ("daily", "اگر بارش نہ ہوئی تو ہم پکنک پر ضرور جائیں گے۔"),
    ("daily", "مجھے آج دفتر سے جلدی نکلنا ہے۔"),
    ("daily", "ہماری ٹیم نے آخری گیند پر میچ جیت لیا۔"),
    ("daily", "مہربانی کر کے آہستہ بولیں۔"),
    ("facts", "پاکستان کا دارالحکومت اسلام آباد ہے۔"),
    ("facts", "دریائے سندھ پاکستان کا سب سے بڑا دریا ہے۔"),
    ("facts", "قائداعظم محمد علی جناح پاکستان کے بانی تھے۔"),
    ("facts", "اردو پاکستان کی قومی زبان ہے۔"),
    ("facts", "کراچی پاکستان کا سب سے بڑا شہر ہے۔"),
    ("numbers", "میرے پاس 25 کتابیں اور 3 قلم ہیں۔"),
    ("numbers", "سال 2024 میں قیمتوں میں 15 فیصد اضافہ ہوا۔"),
    ("numbers", "اس کمرے کا کرایہ 12,500 روپے مہینہ ہے۔"),
    ("numbers", "ہمارے شہر کی آبادی تقریباً 3500000 ہے۔"),
    ("numbers", "وہ 7 بجے گھر پہنچا۔"),
    ("loanwords", "یہ کمپیوٹر پروگرام موبائل فون پر بھی بہت تیزی سے چلتا ہے۔"),
    ("loanwords", "میں نے نیا لیپ ٹاپ خریدا ہے۔"),
    ("loanwords", "انٹرنیٹ کی رفتار آج بہت سست ہے۔"),
    ("loanwords", "ڈاکٹر صاحب نے مجھے دوا لینے کا مشورہ دیا۔"),
    ("loanwords", "بس اسٹاپ اسکول کے بالکل سامنے ہے۔"),
    ("questions", "آپ کل کہاں تھے؟"),
    ("questions", "یہ کتنے کا ہے؟"),
    ("questions", "کیا آپ نے کھانا کھا لیا؟"),
    ("short", "شکریہ۔"),
    ("short", "معاف کیجیے گا۔"),
    ("short", "بہت خوب!"),
    ("short", "جی ہاں، بالکل۔"),
    ("daily", "دکان صبح نو بجے کھلتی ہے اور رات آٹھ بجے بند ہو جاتی ہے۔"),
    ("daily", "استاد نے طلبہ کو بتایا کہ کل امتحان ہوگا۔"),
    ("daily", "ہمیں اپنے ماحول کو صاف رکھنا چاہیے۔"),
    ("long", "محکمہ موسمیات کے مطابق اگلے تین دنوں میں لاہور، کراچی اور پشاور میں تیز بارش کا امکان ہے، اس لیے شہریوں کو احتیاط کرنے کی ہدایت کی گئی ہے۔"),
    ("long", "علم حاصل کرنا ہر انسان کے لیے ضروری ہے کیونکہ علم کے بغیر نہ کوئی قوم ترقی کر سکتی ہے اور نہ کوئی معاشرہ آگے بڑھ سکتا ہے۔"),
    ("long", "جب میں چھوٹا تھا تو ہر گرمیوں کی چھٹیوں میں اپنے نانا کے گاؤں جایا کرتا تھا، جہاں ہم سارا دن آم کے درختوں کے نیچے کھیلتے تھے۔"),
    ("long", "وقت پر کام کرنا کامیابی کی کنجی ہے اور جو لوگ آج کا کام کل پر چھوڑ دیتے ہیں وہ اکثر پیچھے رہ جاتے ہیں۔"),
]

def make_item(text, set_name, category, human=None):
    spoken = normalize(text)
    ipa = phonemize(spoken)
    phones = phones_for(ipa, VOCAB)
    if not 2 <= len(phones) <= MAX_TOKENS:
        raise ValueError(f"{len(phones)} phones is outside 2-{MAX_TOKENS}")
    return {"set": set_name, "category": category, "text": text, "spoken": spoken, "ipa": ipa,
            "phones": phones, "ids": [VOCAB[p] for p in phones], "human": human}

ITEMS, SKIPPED = [], {}

def skip(reason):
    SKIPPED[reason] = SKIPPED.get(reason, 0) + 1

for category, text in EVERYDAY[:8] if SMOKE_TEST else EVERYDAY:
    try:
        ITEMS.append(make_item(text, "everyday", category))
    except ValueError as error:
        skip(f"everyday: {error}")

if EVAL_HUMAN or CORPUS_SENTENCES:
    source = prepare_data.rows("test")
    taken = 0
    try:
        for shard, index, script, _ipa, audio in source:
            if not script or not audio or not audio["bytes"]:
                continue
            try:
                waveform, rate = sf.read(io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True)
                if not 1.0 <= len(waveform) / rate <= 12:
                    skip("corpus: recording outside 1-12 s")
                    continue
                divisor = gcd(rate, SAMPLE_RATE)
                human = resample_poly(waveform.mean(axis=1), SAMPLE_RATE // divisor, rate // divisor).astype("float32")
                ITEMS.append(make_item(script, "corpus", "corpus", human))
            except ValueError as error:
                skip(f"corpus: {str(error)[:60]}")
                continue
            taken += 1
            if taken == CORPUS_SENTENCES:
                break
    finally:
        source.close()

for number, item in enumerate(ITEMS):
    item["id"] = f"{item['set']}-{number:03d}"
print(f"{len(ITEMS)} sentences: " + ", ".join(
    f"{name} {sum(i['set'] == name for i in ITEMS)}" for name in ("corpus", "everyday")))
if SKIPPED:
    print("Skipped:", json.dumps(SKIPPED, ensure_ascii=False))
if not ITEMS:
    raise RuntimeError("No usable test sentences")

## Synthesize with each system

* **nano** uses the int8 C runtime with the same noise seeding as the training
  project's tools. It also reports the real-time factor (compute time ÷ audio time)
  on this machine's CPU, single thread.
* **teacher** is MMS-TTS Urdu (pinned revision) at the noise scale used to make the
  training data, trimmed of leading and trailing silence.
* **human** is the corpus recording, resampled to 24 kHz.

Every system's audio goes through the same 24 kHz → 16 kHz resampling before Whisper.

In [ ]:
import torch

def trim_silence(wave, rate, threshold_db=-40, pad_seconds=0.08):
    frame = int(rate * 0.01)
    energy = np.sqrt(np.convolve(wave ** 2, np.ones(frame) / frame, mode="same"))
    loud = np.flatnonzero(energy > energy.max() * 10 ** (threshold_db / 20))
    if not len(loud):
        return wave
    pad = int(rate * pad_seconds)
    return wave[max(0, loud[0] - pad):loud[-1] + pad]

AUDIO = {"nano": {}, "teacher": {}, "human": {}}
TIMING = {}

started = time.time()
for item in ITEMS:
    seed = int.from_bytes(hashlib.sha256(" ".join(item["phones"]).encode()).digest()[:8], "big")
    begin = time.perf_counter()
    wave = NANO.say(item["ids"], seed)
    TIMING[item["id"]] = (time.perf_counter() - begin) / (len(wave) / SAMPLE_RATE)
    AUDIO["nano"][item["id"]] = np.clip(wave, -1, 1).astype("float32")
rtf = np.array(list(TIMING.values()))
print(f"nano: {len(ITEMS)} sentences in {time.time() - started:.0f} s; "
      f"real-time factor median {np.median(rtf):.3f}, mean {rtf.mean():.3f} (lower is faster; <1 is real time)")

if EVAL_HUMAN:
    for item in ITEMS:
        if item["human"] is not None:
            AUDIO["human"][item["id"]] = np.clip(item["human"], -1, 1)
    print(f"human: {len(AUDIO['human'])} recordings")

if EVAL_TEACHER:
    from transformers import AutoTokenizer, VitsModel
    TEACHER = "facebook/mms-tts-urd-script_arabic"
    TEACHER_REVISION = "4c4f53d52e4f0b40cb8e7f36dcef66a2a73619c5"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(TEACHER, revision=TEACHER_REVISION)
    teacher = VitsModel.from_pretrained(TEACHER, revision=TEACHER_REVISION).to(device).eval()
    teacher.noise_scale, teacher.noise_scale_duration = TEACHER_NOISE_SCALE, 0.0
    divisor = gcd(teacher.config.sampling_rate, SAMPLE_RATE)
    for number, item in enumerate(ITEMS):
        inputs = tokenizer(item["spoken"], return_tensors="pt").to(device)
        torch.manual_seed(number)
        with torch.no_grad():
            wave = teacher(**inputs).waveform[0].float().cpu().numpy()
        wave = trim_silence(wave, teacher.config.sampling_rate)
        wave = resample_poly(wave, SAMPLE_RATE // divisor, teacher.config.sampling_rate // divisor)
        AUDIO["teacher"][item["id"]] = np.clip(wave, -1, 1).astype("float32")
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"teacher: {len(AUDIO['teacher'])} sentences")

SYSTEMS = [name for name in ("nano", "teacher", "human") if AUDIO[name]]

## Transcribe with Whisper

Whisper is told the language is Urdu and the task is transcription, so it never
translates. All clips are 16 kHz mono. Decoding is greedy by default
(`WHISPER_BEAMS = 1`) and deterministic for a given GPU and library version.

In [ ]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
if device == -1 and WHISPER_MODEL.endswith("large-v3"):
    print("Warning: no GPU found; large-v3 will be very slow. Enable a GPU for the full run.")
asr = pipeline("automatic-speech-recognition", model=WHISPER_MODEL, device=device,
               torch_dtype=torch.float16 if device == 0 else torch.float32)
GENERATE = {"language": "urdu", "task": "transcribe", "num_beams": WHISPER_BEAMS}

def to_16k(wave):
    return resample_poly(wave, 2, 3).astype("float32")  # 24 kHz -> 16 kHz

HYPOTHESES = {}
started = time.time()
for system in SYSTEMS:
    ids = list(AUDIO[system])
    clips = [{"raw": to_16k(AUDIO[system][i]), "sampling_rate": 16_000} for i in ids]
    outputs = asr(clips, batch_size=WHISPER_BATCH, chunk_length_s=30, generate_kwargs=GENERATE)
    HYPOTHESES[system] = {i: out["text"].strip() for i, out in zip(ids, outputs)}
    print(f"{system}: transcribed {len(ids)} clips ({time.time() - started:.0f} s elapsed)")
del asr
if device == 0:
    torch.cuda.empty_cache()

# A first look
first = ITEMS[0]
print("\nspoken :", first["spoken"])
for system in SYSTEMS:
    if first["id"] in HYPOTHESES[system]:
        print(f"{system:8s}:", HYPOTHESES[system][first["id"]])

## Score

Both sides are put through the same cleanup before comparing: numbers are spelled
out in Urdu words (Whisper often writes digits), punctuation and vowel marks are
removed, and the Arabic look-alike letters (ي ك ه) are mapped to their Urdu forms.

* **WER** is computed over words; **CER** over characters *without spaces*, because
  Urdu word spacing is inconsistent (e.g. دارالحکومت / دار الحکومت) and would
  otherwise count as errors even when every sound is right.
* Scores are **corpus-level**: total errors ÷ total reference words (or characters),
  with a 95% bootstrap interval over sentences.
* *Exact* is the share of sentences transcribed perfectly.
* *runaway* counts transcripts more than twice the reference length (+3 words), which is
  almost always Whisper looping on a word. They still count as errors, but a few of them
  can swing WER a lot, so the last column shows WER without them.

In [ ]:
import unicodedata
import pandas as pd
from urdu_frontend import DIGIT_MAP, NUMBER, speak_number

DIACRITICS = re.compile("[ؐ-ًؚ-ٰٟۖ-ۭ]")
LETTER_FIX = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ه": "ہ", "ة": "ہ", "ۀ": "ہ", "ە": "ہ", "ۃ": "ہ",
                            "ٱ": "ا", "أ": "ا", "إ": "ا", "ۓ": "ے", "ې": "ے"})

def canon(text):
    """The cleanup applied to both the reference and the transcript."""
    text = unicodedata.normalize("NFKC", text).translate(DIGIT_MAP)
    try:
        text = NUMBER.sub(lambda match: speak_number(match, False), text)
    except ValueError:
        text = re.sub(r"[0-9,.%]+", " ", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Cf" and c != "ـ")
    text = DIACRITICS.sub("", text).translate(LETTER_FIX)
    text = "".join(c if unicodedata.category(c).startswith("L") else " " for c in text)
    return re.sub(r"\s+", " ", text).strip()

def edit_ops(ref, hyp):
    """(substitutions, deletions, insertions) of the minimum edit between two sequences."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]))
    i, j, s, dele, ins = n, m, 0, 0, 0
    while i or j:
        if i and j and d[i][j] == d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            s += ref[i - 1] != hyp[j - 1]
            i, j = i - 1, j - 1
        elif i and d[i][j] == d[i - 1][j] + 1:
            dele, i = dele + 1, i - 1
        else:
            ins, j = ins + 1, j - 1
    return s, dele, ins

rows = []
for item in ITEMS:
    reference = canon(item["spoken"])
    ref_words, ref_chars = reference.split(), list(reference.replace(" ", ""))
    for system in SYSTEMS:
        if item["id"] not in HYPOTHESES[system]:
            continue
        hypothesis = canon(HYPOTHESES[system][item["id"]])
        hyp_words, hyp_chars = hypothesis.split(), list(hypothesis.replace(" ", ""))
        ws, wd, wi = edit_ops(ref_words, hyp_words)
        cs, cd, ci = edit_ops(ref_chars, hyp_chars)
        rows.append({
            "id": item["id"], "set": item["set"], "category": item["category"], "system": system,
            "text": item["text"], "reference": reference, "whisper": HYPOTHESES[system][item["id"]],
            "hypothesis": hypothesis, "seconds": round(len(AUDIO[system][item["id"]]) / SAMPLE_RATE, 2),
            "words": len(ref_words), "w_sub": ws, "w_del": wd, "w_ins": wi, "w_err": ws + wd + wi,
            "chars": len(ref_chars), "c_err": cs + cd + ci,
            # Whisper sometimes loops on one word; a transcript far longer than the reference is flagged.
            "runaway": len(hyp_words) > 2 * len(ref_words) + 3,
        })
RESULTS = pd.DataFrame(rows)
RESULTS["wer"] = RESULTS.w_err / RESULTS.words
RESULTS["cer"] = RESULTS.c_err / RESULTS.chars
print(len(RESULTS), "scored transcripts")

In [ ]:
rng = np.random.default_rng(SEED)
BOOTSTRAPS = 2000

def rate(frame, errors, total):
    return frame[errors].sum() / max(1, frame[total].sum())

def interval(frame, errors, total):
    err, tot = frame[errors].to_numpy(), frame[total].to_numpy()
    draws = rng.integers(0, len(frame), (BOOTSTRAPS, len(frame)))
    values = err[draws].sum(1) / np.maximum(1, tot[draws].sum(1))
    return np.percentile(values, [2.5, 97.5])

def summarize(frame):
    wer_lo, wer_hi = interval(frame, "w_err", "words")
    cer_lo, cer_hi = interval(frame, "c_err", "chars")
    return {
        "sentences": len(frame), "words": int(frame.words.sum()),
        "WER %": 100 * rate(frame, "w_err", "words"), "WER lo": 100 * wer_lo, "WER hi": 100 * wer_hi,
        "CER %": 100 * rate(frame, "c_err", "chars"), "CER lo": 100 * cer_lo, "CER hi": 100 * cer_hi,
        "sub %": 100 * frame.w_sub.sum() / max(1, frame.words.sum()),
        "del %": 100 * frame.w_del.sum() / max(1, frame.words.sum()),
        "ins %": 100 * frame.w_ins.sum() / max(1, frame.words.sum()),
        "exact %": 100 * (frame.w_err == 0).mean(),
        "runaway": int(frame.runaway.sum()),
        "WER excl. runaway %": 100 * rate(frame[~frame.runaway], "w_err", "words") if (~frame.runaway).any() else float("nan"),
    }

SUMMARY = pd.DataFrame({(s, y): summarize(g) for (s, y), g in RESULTS.groupby(["set", "system"])}).T
SUMMARY.index.names = ["set", "system"]
order = {"human": 0, "teacher": 1, "nano": 2}
SUMMARY = SUMMARY.sort_index(key=lambda idx: idx.map(order) if idx.name == "system" else idx)
display(SUMMARY[["sentences", "words", "WER %", "WER lo", "WER hi", "CER %", "CER lo", "CER hi", "exact %",
                 "runaway", "WER excl. runaway %"]]
        .style.format({c: "{:.1f}" for c in SUMMARY.columns if c not in ("sentences", "words", "runaway")}))
if RESULTS.runaway.any():
    print(f"\n{int(RESULTS.runaway.sum())} transcript(s) looped or ran far past the reference length "
          "(Whisper hallucination, not the voice). They count as errors above; the last column shows WER without them.")

# Paired comparison: the same sentences, nano minus teacher / human.
PAIRED = {}
for set_name, group in RESULTS.groupby("set"):
    wide = group.pivot(index="id", columns="system")
    for other in ("teacher", "human"):
        if "nano" in group.system.values and other in group.system.values:
            both = wide.dropna(subset=[("w_err", other), ("w_err", "nano")])
            if not len(both):
                continue
            err = both["w_err"][["nano", other]].to_numpy()
            words = both["words"]["nano"].to_numpy()
            draws = rng.integers(0, len(both), (BOOTSTRAPS, len(both)))
            diff = (err[draws, 0].sum(1) - err[draws, 1].sum(1)) / np.maximum(1, words[draws].sum(1))
            PAIRED[f"{set_name}: nano - {other}"] = {
                "WER points": 100 * (err[:, 0].sum() - err[:, 1].sum()) / words.sum(),
                "lo": 100 * np.percentile(diff, 2.5), "hi": 100 * np.percentile(diff, 97.5)}
if PAIRED:
    print("\nNano WER minus the other system's WER on the same sentences (percentage points, 95% interval).")
    print("Zero inside the interval means no measurable difference.")
    display(pd.DataFrame(PAIRED).T.style.format("{:+.1f}"))
print(f"\nNano real-time factor on this CPU: median {np.median(rtf):.3f} "
      f"(i.e. {1 / np.median(rtf):.0f}x faster than real time, one thread)")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

COLORS = {"human": "#8a8a8a", "teacher": "#c58a2b", "nano": "#0b6b4f"}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)
sets = [s for s in ("corpus", "everyday") if s in SUMMARY.index.get_level_values("set")]
for axis, metric in zip(axes, ("WER", "CER")):
    width = 0.8 / max(1, len(SYSTEMS))
    for k, system in enumerate(SYSTEMS):
        xs, vals, errs = [], [], [[], []]
        for p, set_name in enumerate(sets):
            if (set_name, system) in SUMMARY.index:
                row = SUMMARY.loc[(set_name, system)]
                xs.append(p + (k - (len(SYSTEMS) - 1) / 2) * width)
                vals.append(row[f"{metric} %"])
                errs[0].append(row[f"{metric} %"] - row[f"{metric} lo"])
                errs[1].append(row[f"{metric} hi"] - row[f"{metric} %"])
        axis.bar(xs, vals, width * 0.92, yerr=errs, color=COLORS[system], label=system, capsize=3)
    axis.set_xticks(range(len(sets)), sets)
    axis.set_ylabel(f"{metric} % (lower is better)")
    axis.set_title(f"Whisper {metric} by test set")
    axis.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
fig.tight_layout()
CHART = OUT / "wer-cer.png"
fig.savefig(CHART, dpi=130)
plt.show()

if "everyday" in sets:
    print("\nWER % on the everyday sentences, by category")
    by_category = (RESULTS[RESULTS.set == "everyday"]
                   .groupby(["category", "system"]).apply(lambda g: 100 * g.w_err.sum() / max(1, g.words.sum()), include_groups=False)
                   .unstack("system").reindex(columns=[s for s in ("teacher", "nano") if s in SYSTEMS]))
    by_category["sentences"] = RESULTS[(RESULTS.set == "everyday") & (RESULTS.system == "nano")].groupby("category").size()
    display(by_category.style.format({c: "{:.1f}" for c in by_category.columns if c != "sentences"}))

## Where nano is worst

The sentences where nano's WER is furthest above the teacher's (or, with no teacher,
its own highest WER). Whisper's transcript is shown for every system so you can see
whether the model mispronounced something or Whisper simply misheard it: if the
**teacher and human** transcripts are also wrong, it is Whisper's difficulty with
that sentence, not the voice.

In [ ]:
from IPython.display import Audio, display

wide = RESULTS.pivot(index="id", columns="system", values="wer")
if "teacher" in wide:
    wide["gap"] = wide["nano"] - wide["teacher"]
else:
    wide["gap"] = wide["nano"]
WORST = wide.sort_values("gap", ascending=False).head(12).index.tolist()
by_id = {item["id"]: item for item in ITEMS}
for item_id in WORST:
    item = by_id[item_id]
    print(f"\n[{item_id}] {item['set']}/{item['category']}")
    print("  spoken :", canon(item["spoken"]))
    for system in SYSTEMS:
        if item_id in HYPOTHESES[system]:
            wer = wide.loc[item_id, system]
            print(f"  {system:8s}: {canon(HYPOTHESES[system][item_id])}   (WER {100 * wer:.0f}%)")
    display(Audio(AUDIO["nano"][item_id], rate=SAMPLE_RATE))

## Save the results

The ZIP holds the summary, every transcript and its score, the chart, and the audio
of the worst sentences (all systems), so you can listen to exactly what Whisper heard.

In [ ]:
RESULTS.to_csv(OUT / "results.csv", index=False, encoding="utf-8-sig")
summary = {
    "notebook": NOTEBOOK_VERSION, "source_digest": SOURCE_DIGEST,
    "asr": {"model": WHISPER_MODEL, "beams": WHISPER_BEAMS, "language": "urdu"},
    "voice": {"path": str(VOICE), "weights_sha256_16": VOICE_SHA, "phones": len(VOCAB)},
    "teacher": {"model": "facebook/mms-tts-urd-script_arabic", "noise_scale": TEACHER_NOISE_SCALE} if EVAL_TEACHER else None,
    "corpus": "kaab4321/UrduTTS test split (CC BY-NC 4.0)",
    "skipped": SKIPPED,
    "nano_real_time_factor": {"median": float(np.median(rtf)), "mean": float(rtf.mean())},
    "scores": {f"{s}/{y}": {k: (round(float(v), 3) if isinstance(v, (float, np.floating)) else int(v))
                            for k, v in row.items()} for (s, y), row in SUMMARY.iterrows()},
    "paired_wer_points": {k: {a: round(float(b), 2) for a, b in v.items()} for k, v in PAIRED.items()},
    "worst_sentences": WORST,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

archive = WORKING / f"nano-urdu-eval-{RUN_NAME}.zip"
with tempfile.TemporaryDirectory(dir=WORKING) as temp:
    pending = Path(temp) / archive.name
    with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for name in ("summary.json", "results.csv", "wer-cer.png"):
            bundle.write(OUT / name, name)
        for item_id in WORST:
            for system in SYSTEMS:
                if item_id in AUDIO[system]:
                    wav = Path(temp) / f"{item_id}-{system}.wav"
                    sf.write(wav, AUDIO[system][item_id], SAMPLE_RATE, subtype="PCM_16")
                    bundle.write(wav, f"worst/{item_id}/{system}.wav")
    pending.replace(archive)
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print(f"{archive} ({archive.stat().st_size:,} bytes)")